In [2]:
import torch

In [3]:
epsilon = 1.0
r0 = 0.1

In [4]:
get_x = lambda: torch.tensor([0.1542, 0.8827, 0.3679, 0.2300, 0.5687, 0.0396, 0.5161, 0.0529, 0.9302,
        0.9654, 0.3532, 0.2273, 0.9690, 0.9541, 0.8267, 0.5761, 0.1615, 0.8755,
        0.6661, 0.7874, 0.7718])
d = get_x().shape[0]
d

21

In [5]:
def value_point(point):
    point = point.reshape(-1,3)
    m = point.shape[0]
    result = 0.0
    r = lambda i, j: torch.linalg.vector_norm(point[i, :]-point[j, :])
    for i in range(m):
      for j in range(i+1,m):
        rij = r0/r(i,j)
        result = result + 4 * epsilon * (rij**12-rij**6)
    return result

x = get_x()
value_point(x)

tensor(-0.5429)

## AD grad

In [6]:
x = get_x().unsqueeze(dim=0)
x.requires_grad_(True)
u = value_point(x).unsqueeze(dim=0).unsqueeze(dim=0)
grad_u = torch.autograd.grad(
    inputs=x,
    outputs=u,
    grad_outputs=torch.ones_like(u),
    create_graph=True,
    retain_graph=True,
)[0]
spatial_laplace_u = []
for i in range(d):
    hess_row = torch.autograd.grad(
        inputs=x,
        outputs=grad_u[:,i].sum(),
        grad_outputs=torch.tensor(1.0),
        create_graph=True,
        retain_graph=True
    )[0]
    spatial_laplace_u.append(hess_row[:,i:i+1])
spatial_laplace_u = torch.cat(spatial_laplace_u, dim=1)
grad_u

tensor([[-1.3169e-03,  3.8306e-03,  3.5812e-03,  7.3281e-04, -3.6869e-03,
         -4.0101e-03, -8.5463e+00, -1.5469e+01,  7.7914e+00,  3.0262e-04,
         -1.4624e-04, -3.0497e-04,  3.2267e-02,  1.7847e-02,  5.9078e-03,
          8.5461e+00,  1.5468e+01, -7.7911e+00, -3.1778e-02, -1.6995e-02,
         -5.5132e-03]], grad_fn=<ViewBackward0>)

In [7]:
x = get_x()
x.requires_grad_(True)
u, vjp_fn = torch.func.vjp(value_point, x)
grad_u = vjp_fn(torch.ones_like(u))
grad_u

(tensor([-1.3169e-03,  3.8306e-03,  3.5812e-03,  7.3281e-04, -3.6869e-03,
         -4.0101e-03, -8.5463e+00, -1.5469e+01,  7.7914e+00,  3.0262e-04,
         -1.4624e-04, -3.0497e-04,  3.2267e-02,  1.7847e-02,  5.9078e-03,
          8.5461e+00,  1.5468e+01, -7.7911e+00, -3.1778e-02, -1.6995e-02,
         -5.5132e-03], grad_fn=<ViewBackward0>),)

## analytic grad

In [8]:
x = get_x()
X = x.reshape(-1,3)
X.shape

torch.Size([7, 3])

In [9]:
m = 7
r = lambda a, b: torch.linalg.vector_norm(a-b)
print("i k l V_der")
for i in range(21):
    k = i // 3
    l = i - 3*k
    assert i == 3*k + l
    V_der = 0.0
    for j in range(m):
        if j == k:
            continue
        r_kj = r(X[k,:], X[j,:])
        term = ( -12 * r0**12 * r_kj**(-14) + 6 * r0**6  * r_kj**(-8) ) * ( X[k,l] - X[j,l] )
        V_der += term
    V_der = 4*epsilon * V_der
    print(i, k, l, V_der)

i k l V_der
0 0 0 tensor(-0.0013)
1 0 1 tensor(0.0038)
2 0 2 tensor(0.0036)
3 1 0 tensor(0.0007)
4 1 1 tensor(-0.0037)
5 1 2 tensor(-0.0040)
6 2 0 tensor(-8.5463)
7 2 1 tensor(-15.4690)
8 2 2 tensor(7.7914)
9 3 0 tensor(0.0003)
10 3 1 tensor(-0.0001)
11 3 2 tensor(-0.0003)
12 4 0 tensor(0.0323)
13 4 1 tensor(0.0178)
14 4 2 tensor(0.0059)
15 5 0 tensor(8.5461)
16 5 1 tensor(15.4681)
17 5 2 tensor(-7.7911)
18 6 0 tensor(-0.0318)
19 6 1 tensor(-0.0170)
20 6 2 tensor(-0.0055)


## AD second der

In [10]:
spatial_laplace_u

tensor([[ 6.3579e-03, -3.1801e-02, -3.7733e-02,  8.2562e-03, -3.2133e-02,
         -3.6738e-02, -1.2835e+00, -3.2841e+02,  2.2986e+01, -1.2844e-03,
         -5.0407e-04, -1.8892e-03, -5.2994e-01, -8.7046e-02,  8.5145e-02,
         -1.2831e+00, -3.2841e+02,  2.2986e+01, -5.3161e-01, -9.3006e-02,
          8.3845e-02]], grad_fn=<CatBackward0>)

In [11]:
m = 7
r = lambda a, b: torch.linalg.vector_norm(a-b)
print("i k l V_der")
for i in range(21):
    k = i // 3
    l = i - 3*k
    assert i == 3*k + l
    V_der = 0.0
    for j in range(m):
        if j == k:
            continue
        r_kj = r(X[k,:], X[j,:])
        term1 = ( 12 * 13 * r0**12 * r_kj**(-14) - 6 * 7 *r0**6  * r_kj**(-8) ) * (( X[k,l] - X[j,l] ) / r_kj)**2 
        term2 = ( - 12 * r0**12 * r_kj**(-14) + 6 * r0**6  * r_kj**(-8) ) * ( 1 - (( X[k,l] - X[j,l] ) / r_kj)**2 )
        term  = term1 + term2
        V_der += term
    V_der = 4*epsilon * V_der
    print(i, k, l, V_der)

i k l V_der
0 0 0 tensor(0.0064)
1 0 1 tensor(-0.0318)
2 0 2 tensor(-0.0377)
3 1 0 tensor(0.0083)
4 1 1 tensor(-0.0321)
5 1 2 tensor(-0.0367)
6 2 0 tensor(-1.2834)
7 2 1 tensor(-328.4093)
8 2 2 tensor(22.9859)
9 3 0 tensor(-0.0013)
10 3 1 tensor(-0.0005)
11 3 2 tensor(-0.0019)
12 4 0 tensor(-0.5299)
13 4 1 tensor(-0.0870)
14 4 2 tensor(0.0851)
15 5 0 tensor(-1.2830)
16 5 1 tensor(-328.4138)
17 5 2 tensor(22.9858)
18 6 0 tensor(-0.5316)
19 6 1 tensor(-0.0930)
20 6 2 tensor(0.0838)


---

## vectorize curs

In [12]:
x = get_x().unsqueeze(dim=0)
Y = torch.cat([x, torch.randn(1_000_000, 21)], dim=0)


B = Y.shape[0]
X = Y.view(B, 7, 3)      # (B, 7, 3)
m = X.shape[1]

# pairwise differences: (B, m, m, 3)
dX = X[:, :, None, :] - X[:, None, :, :]

# pairwise distances: (B, m, m)
r = torch.linalg.vector_norm(dX, dim=-1)

# mask out self-interactions
eye = torch.eye(m, dtype=torch.bool, device=X.device).expand(B, m, m)
r = r.masked_fill(eye, float('inf'))

coeff = (
    -12 * r0**12 * r**(-14) +
     6 * r0**6  * r**(-8)
)  # (B, m, m)

force_components = (coeff[..., None] * dX).sum(dim=2)   # (B, m, 3)
V_der_batch = 4 * epsilon * force_components            # (B, 7, 3)
V_der_batch_flat = V_der_batch.view(B, 21)              # (B, 21)
V_der_batch_flat[0]

tensor([-1.3169e-03,  3.8306e-03,  3.5812e-03,  7.3281e-04, -3.6869e-03,
        -4.0101e-03, -8.5463e+00, -1.5469e+01,  7.7914e+00,  3.0262e-04,
        -1.4624e-04, -3.0497e-04,  3.2267e-02,  1.7847e-02,  5.9078e-03,
         8.5461e+00,  1.5468e+01, -7.7911e+00, -3.1778e-02, -1.6995e-02,
        -5.5132e-03])

In [13]:
grad_u

(tensor([-1.3169e-03,  3.8306e-03,  3.5812e-03,  7.3281e-04, -3.6869e-03,
         -4.0101e-03, -8.5463e+00, -1.5469e+01,  7.7914e+00,  3.0262e-04,
         -1.4624e-04, -3.0497e-04,  3.2267e-02,  1.7847e-02,  5.9078e-03,
          8.5461e+00,  1.5468e+01, -7.7911e+00, -3.1778e-02, -1.6995e-02,
         -5.5132e-03], grad_fn=<ViewBackward0>),)

In [14]:
x = get_x().unsqueeze(dim=0)
Y = torch.cat([x, torch.randn(1_000_000, 21)], dim=0)


# Y: (B, 21)  ->  X: (B, 7, 3)
B = Y.shape[0]
m = 7
X = Y.view(B, m, 3)                       # (B, 7, 3)

# pairwise differences: dX[b, k, j, l] = X[b, k, l] - X[b, j, l]
dX = X[:, :, None, :] - X[:, None, :, :]  # (B, m, m, 3)

# pairwise distances r[b, k, j]
r = torch.linalg.vector_norm(dX, dim=-1)  # (B, m, m)

# mask self-interactions (k == j): r -> inf so powers -> 0, ratios -> 0
eye = torch.eye(m, dtype=torch.bool, device=X.device).expand(B, m, m)
r = r.masked_fill(eye, float('inf'))      # (B, m, m)

# coefficients that depend only on r_kj
coeff1 =  12 * 13 * r0**12 * r**(-14) - 6 * 7 * r0**6 * r**(-8)   # (B, m, m)
coeff2 = -12      * r0**12 * r**(-14) + 6     * r0**6 * r**(-8)   # (B, m, m)

# (X[k,l] - X[j,l]) / r_kj, squared, per component l
ratio      = dX / r[..., None]            # (B, m, m, 3)
ratio_sq   = ratio**2                     # (B, m, m, 3)

# term1, term2, term per (b, k, j, l)
term1 = coeff1[..., None] * ratio_sq                 # (B, m, m, 3)
term2 = coeff2[..., None] * (1.0 - ratio_sq)         # (B, m, m, 3)
term  = term1 + term2                                # (B, m, m, 3)

# sum over j to get V_der[b, k, l]
V_der = term.sum(dim=2)                  # (B, m, 3)

# scale by 4 * epsilon, matching your code
V_der = 4 * epsilon * V_der              # (B, 7, 3)

# if you want the flattened 21-vector per batch element:
V_der_flat = V_der.view(B, 21)           # (B, 21)
V_der_flat[0]

tensor([ 6.3579e-03, -3.1801e-02, -3.7733e-02,  8.2562e-03, -3.2133e-02,
        -3.6738e-02, -1.2834e+00, -3.2841e+02,  2.2986e+01, -1.2844e-03,
        -5.0407e-04, -1.8892e-03, -5.2994e-01, -8.7046e-02,  8.5145e-02,
        -1.2830e+00, -3.2841e+02,  2.2986e+01, -5.3161e-01, -9.3006e-02,
         8.3845e-02])

In [15]:
spatial_laplace_u

tensor([[ 6.3579e-03, -3.1801e-02, -3.7733e-02,  8.2562e-03, -3.2133e-02,
         -3.6738e-02, -1.2835e+00, -3.2841e+02,  2.2986e+01, -1.2844e-03,
         -5.0407e-04, -1.8892e-03, -5.2994e-01, -8.7046e-02,  8.5145e-02,
         -1.2831e+00, -3.2841e+02,  2.2986e+01, -5.3161e-01, -9.3006e-02,
          8.3845e-02]], grad_fn=<CatBackward0>)

In [ ]:
# compute both at once
x = get_x().unsqueeze(dim=0)
bs = 1_000_000
Y = torch.cat([x, torch.randn(bs-1, 21)], dim=0)


# Y: (B, 21)  ->  X: (B, 7, 3)
B = Y.shape[0]
m = 7
X = Y.view(B, m, 3)                        # (B, 7, 3)

# pairwise differences and distances
dX = X[:, :, None, :] - X[:, None, :, :]   # (B, m, m, 3)
r  = torch.linalg.vector_norm(dX, dim=-1)  # (B, m, m)

# mask self-interactions
eye = torch.eye(m, dtype=torch.bool, device=X.device).expand(B, m, m)
r = r.masked_fill(eye, float('inf'))       # (B, m, m)

# shared powers of r
r_inv8  = r**(-8)                          # (B, m, m)
r_inv14 = r**(-14)                         # (B, m, m)

# ---------------------------
# Gradient of LJ potential
# ---------------------------
# coeff_grad[k, j] multiplies (X[k,l] - X[j,l])
coeff_grad = -12 * r0**12 * r_inv14 + 6 * r0**6 * r_inv8    # (B, m, m)

grad_pair = coeff_grad[..., None] * dX                      # (B, m, m, 3)
grad = grad_pair.sum(dim=2)                                 # (B, m, 3)
grad = 4 * epsilon * grad                                   # (B, 7, 3)

# ---------------------------
# Laplacian of LJ potential
# ---------------------------
# reuse dX, r, r_inv8, r_inv14

coeff1 =  12 * 13 * r0**12 * r_inv14 - 6 * 7 * r0**6 * r_inv8   # (B, m, m)
coeff2 = -12      * r0**12 * r_inv14 + 6     * r0**6 * r_inv8   # (B, m, m)

ratio    = dX / r[..., None]                 # (B, m, m, 3)
ratio_sq = ratio**2                          # (B, m, m, 3)

term1 = coeff1[..., None] * ratio_sq                     # (B, m, m, 3)
term2 = coeff2[..., None] * (1.0 - ratio_sq)             # (B, m, m, 3)
lap_pair = term1 + term2                                 # (B, m, m, 3)

laplace = lap_pair.sum(dim=2)                            # (B, m, 3)
laplace = 4 * epsilon * laplace                          # (B, 7, 3)

# (optional) flattened versions
grad_flat    = grad.view(B, 21)      # (B, 21)
laplace_flat = laplace.view(B, 21)   # (B, 21)

grad_flat[0], laplace_flat[0]

(tensor([-1.3169e-03,  3.8306e-03,  3.5812e-03,  7.3281e-04, -3.6869e-03,
         -4.0101e-03, -8.5463e+00, -1.5469e+01,  7.7914e+00,  3.0262e-04,
         -1.4624e-04, -3.0497e-04,  3.2267e-02,  1.7847e-02,  5.9078e-03,
          8.5461e+00,  1.5468e+01, -7.7911e+00, -3.1778e-02, -1.6995e-02,
         -5.5132e-03]),
 tensor([ 6.3579e-03, -3.1801e-02, -3.7733e-02,  8.2562e-03, -3.2133e-02,
         -3.6738e-02, -1.2834e+00, -3.2841e+02,  2.2986e+01, -1.2844e-03,
         -5.0407e-04, -1.8892e-03, -5.2994e-01, -8.7046e-02,  8.5145e-02,
         -1.2830e+00, -3.2841e+02,  2.2986e+01, -5.3161e-01, -9.3006e-02,
          8.3845e-02]))

In [17]:
def lj_grad_and_laplace_batched(Y, n_atoms, d, r0, epsilon):
    """
    Y: (B, n_atoms * d)
    returns:
        grad:    (B, n_atoms, d)
        laplace: (B, n_atoms, d)
    """
    B = Y.shape[0]
    X = Y.view(B, n_atoms, d)                      # (B, n_atoms, d)

    # pairwise differences: dX[b, k, j, c] = X[b, k, c] - X[b, j, c]
    dX = X[:, :, None, :] - X[:, None, :, :]      # (B, n_atoms, n_atoms, d)

    # pairwise distances r[b, k, j]
    r = torch.linalg.vector_norm(dX, dim=-1)      # (B, n_atoms, n_atoms)

    # mask self-interactions (k == j)
    eye = torch.eye(n_atoms, dtype=torch.bool, device=X.device).expand(B, n_atoms, n_atoms)
    r = r.masked_fill(eye, float('inf'))          # (B, n_atoms, n_atoms)

    # shared powers of r
    r_inv8  = r**(-8)                             # (B, n_atoms, n_atoms)
    r_inv14 = r**(-14)                            # (B, n_atoms, n_atoms)

    # ---------------------------
    # Gradient of LJ potential
    # ---------------------------
    coeff_grad = -12 * r0**12 * r_inv14 + 6 * r0**6 * r_inv8     # (B, n_atoms, n_atoms)

    grad_pair = coeff_grad[..., None] * dX                       # (B, n_atoms, n_atoms, d)
    grad = grad_pair.sum(dim=2)                                  # (B, n_atoms, d)
    grad = 4 * epsilon * grad                                    # (B, n_atoms, d)

    # ---------------------------
    # Laplacian of LJ potential
    # ---------------------------
    coeff1 =  12 * 13 * r0**12 * r_inv14 - 6 * 7 * r0**6 * r_inv8   # (B, n_atoms, n_atoms)
    coeff2 = -12      * r0**12 * r_inv14 + 6     * r0**6 * r_inv8   # (B, n_atoms, n_atoms)

    ratio    = dX / r[..., None]                    # (B, n_atoms, n_atoms, d)
    ratio_sq = ratio**2                             # (B, n_atoms, n_atoms, d)

    term1   = coeff1[..., None] * ratio_sq                      # (B, n_atoms, n_atoms, d)
    term2   = coeff2[..., None] * (1.0 - ratio_sq)              # (B, n_atoms, n_atoms, d)
    lap_pair = term1 + term2                                   # (B, n_atoms, n_atoms, d)

    laplace = lap_pair.sum(dim=2)                               # (B, n_atoms, d)
    laplace = 4 * epsilon * laplace                             # (B, n_atoms, d)

    #return grad, laplace
    
    grad_flat    = grad.view(B, n_atoms * d)
    laplace_flat = laplace.view(B, n_atoms * d)
    return grad_flat, laplace_flat

In [24]:
bs = 1_000
x = get_x().unsqueeze(dim=0)
Y = torch.cat([x, torch.randn(bs-1, 21)], dim=0)
g,l = lj_grad_and_laplace_batched(Y, 7, 3, r0, epsilon)
g[0], l[0]

(tensor([-1.3169e-03,  3.8306e-03,  3.5812e-03,  7.3281e-04, -3.6869e-03,
         -4.0101e-03, -8.5463e+00, -1.5469e+01,  7.7914e+00,  3.0262e-04,
         -1.4624e-04, -3.0497e-04,  3.2267e-02,  1.7847e-02,  5.9078e-03,
          8.5461e+00,  1.5468e+01, -7.7911e+00, -3.1778e-02, -1.6995e-02,
         -5.5132e-03]),
 tensor([ 6.3579e-03, -3.1801e-02, -3.7733e-02,  8.2562e-03, -3.2133e-02,
         -3.6738e-02, -1.2834e+00, -3.2841e+02,  2.2986e+01, -1.2844e-03,
         -5.0407e-04, -1.8892e-03, -5.2994e-01, -8.7046e-02,  8.5145e-02,
         -1.2830e+00, -3.2841e+02,  2.2986e+01, -5.3161e-01, -9.3006e-02,
          8.3845e-02]))


---

## vectorize

In [18]:
X

tensor([[[ 0.1542,  0.8827,  0.3679],
         [ 0.2300,  0.5687,  0.0396],
         [ 0.5161,  0.0529,  0.9302],
         ...,
         [ 0.9690,  0.9541,  0.8267],
         [ 0.5761,  0.1615,  0.8755],
         [ 0.6661,  0.7874,  0.7718]],

        [[ 0.4374, -1.7741,  0.3870],
         [ 1.8950, -1.0056,  0.9073],
         [-0.2292, -1.0128,  0.5870],
         ...,
         [-0.6043, -1.0738,  0.2540],
         [-0.6746, -0.8997,  0.9865],
         [ 0.7185, -0.6984, -0.7224]],

        [[ 1.1074, -1.2022,  0.8111],
         [-0.4333, -1.8407, -0.9472],
         [-1.2129,  0.8661,  1.2117],
         ...,
         [ 0.0918,  0.7563,  0.4586],
         [-0.6073,  1.3756, -1.0739],
         [-0.0429,  0.8987, -0.1885]],

        ...,

        [[ 1.3171,  0.2191,  1.9849],
         [-0.5204, -0.3494,  1.4683],
         [ 0.1712, -0.0555, -0.1123],
         ...,
         [ 0.5236,  0.6712, -0.0333],
         [ 0.7888,  0.2129,  1.2479],
         [ 0.7076,  2.1532, -0.2602]],

        [[

In [19]:
m = 7
r = lambda a, b: torch.linalg.vector_norm(a-b)

print("i k l V_der")
for k in range(m):
    V_der = 0.0
    for j in range(m):
        if j == k:
            continue
        r_kj = r(X[k,:], X[j,:])
        term = ( -12 * r0**12 * r_kj**(-14) + 6 * r0**6  * r_kj**(-8) ) * ( X[k,:] - X[j,:] )
        V_der += term
    V_der = 4*epsilon * V_der
    print(k, V_der)

#print("i k l V_der")
#for i in range(21):
#    k = i // 3
#    l = i - 3*k
#    assert i == 3*k + l
#    V_der = 0.0
#    for j in range(m):
#        if j == k:
#            continue
#        r_kj = r(X[k,:], X[j,:])
#        term1 = ( 12 * 13 * r0**12 * r_kj**(-14) - 6 * 7 *r0**6  * r_kj**(-8) ) * (( X[k,l] - X[j,l] ) / r_kj)**2 
#        term2 = ( - 12 * r0**12 * r_kj**(-14) + 6 * r0**6  * r_kj**(-8) ) * ( 1 - (( X[k,l] - X[j,l] ) / r_kj)**2 )
#        term  = term1 + term2
#        V_der += term
#    V_der = 4*epsilon * V_der
#    print(i, k, l, V_der)

i k l V_der
0 tensor([[ 8.1284e-11,  1.6640e-10,  8.1688e-11],
        [ 3.5418e-11,  1.5809e-10,  6.3868e-11],
        [ 2.6425e-10,  1.5804e-10,  9.7507e-11],
        [ 2.0731e-10,  1.8577e-10,  1.4996e-10],
        [ 2.2042e-10,  1.1103e-10,  1.1199e-10],
        [ 1.6694e-11, -1.5898e-10,  1.9901e-10],
        [ 8.7409e-11,  2.4778e-10,  1.6707e-10]])
1 tensor([[-6.7888e-12, -8.0621e-11,  1.3146e-11],
        [ 8.7628e-11, -5.2130e-11,  8.7559e-11],
        [ 1.6216e-11, -5.9994e-12,  2.5090e-11],
        [-1.2552e-10,  6.0054e-11,  3.4934e-11],
        [-2.6646e-11, -6.7033e-11, -9.7271e-12],
        [-3.3611e-11, -7.1209e-11,  3.8397e-11],
        [ 4.2142e-11, -8.5923e-12, -4.2971e-11]])
2 tensor([[ 1.9366e-10, -1.7051e-10,  1.1255e-10],
        [-6.3970e-11, -2.2188e-10, -1.0284e-10],
        [-6.9037e-11,  2.0394e-10,  8.9657e-11],
        [-7.5734e-11,  5.1669e-12, -6.1520e-11],
        [ 2.4786e-11,  4.5360e-11,  2.8350e-11],
        [-1.5226e-10,  6.2015e-11, -1.4621e-10],
